In [0]:
%run "../../utils/00_config"

In [0]:
from pyspark.sql import functions as F

# Gold source
price_history = spark.table(
    get_gold_table("marts", "mart_item_price_history")
)

print(f"Price history records: {price_history.count():,}")

display(
    price_history
    .groupBy("item_slug", "order_type", "price_quality_status")
    .agg(F.count("*").alias("total_offers"))
    .orderBy("item_slug", "order_type", "price_quality_status")
)

In [0]:
# Aggregate market statistics
market_summary = (
    price_history
    .groupBy(
        "execution_id",
        "item_slug",
        "item_name",
        "product_slug",
        "category",
        "order_type",
        "observed_at"
    )
    .agg(
        # Offer volume
        F.count("*").alias("total_offers"),

        # Price statistics: all eligible offers
        F.min("platinum").alias("min_price"),
        F.max("platinum").alias("max_price"),
        F.avg("platinum").alias("avg_price"),
        F.percentile_approx(
            "platinum", 0.5, 10000
        ).alias("median_price"),
        F.percentile_approx(
            "platinum", 0.25, 10000
        ).alias("p25_price"),
        F.percentile_approx(
            "platinum", 0.75, 10000
        ).alias("p75_price"),

        # Quality metrics
        F.sum(
            F.when(F.col("price_quality_status") == "within_iqr", 1).otherwise(0)
        ).alias("within_iqr_offers"),
        F.sum(
            F.when(F.col("price_quality_status") == "outlier", 1).otherwise(0)
        ).alias("outlier_offers"),
        F.sum(
            F.when(F.col("price_quality_status") == "insufficient_sample", 1).otherwise(0)
        ).alias("insufficient_sample_offers")
    )
)

display(market_summary.orderBy("item_slug", "order_type"))

In [0]:
# Calculate price statistics excluding outliers
within_iqr_summary = (
    price_history
    .filter(F.col("price_quality_status") == "within_iqr")
    .groupBy(
        "execution_id",
        "item_slug",
        "order_type"
    )
    .agg(
        F.min("platinum").alias("within_iqr_min_price"),
        F.max("platinum").alias("within_iqr_max_price"),
        F.avg("platinum").alias("within_iqr_avg_price"),
        F.percentile_approx(
            "platinum", 0.5, 10000
        ).alias("within_iqr_median_price"),
        F.percentile_approx(
            "platinum", 0.25, 10000
        ).alias("within_iqr_p25_price"),
        F.percentile_approx(
            "platinum", 0.75, 10000
        ).alias("within_iqr_p75_price")
    )
)

# Combine general and filtered statistics
market_summary = (
    market_summary
    .join(
        within_iqr_summary,
        on=["execution_id", "item_slug", "order_type"],
        how="left"
    )
)

display(
    market_summary
    .select(
        "item_slug",
        "order_type",
        "total_offers",
        "within_iqr_offers",
        "outlier_offers",
        "avg_price",
        "median_price",
        "within_iqr_avg_price",
        "within_iqr_median_price"
    )
    .orderBy("item_slug", "order_type")
)

In [0]:
# Add price spread indicators
market_summary = (
    market_summary
    .withColumn(
        "market_spread",
        F.col("max_price") - F.col("min_price")
    )
    .withColumn(
        "within_iqr_spread",
        F.col("within_iqr_max_price") - F.col("within_iqr_min_price")
    )
)

display(
    market_summary
    .select(
        "item_slug",
        "order_type",
        "total_offers",
        "min_price",
        "max_price",
        "market_spread",
        "within_iqr_min_price",
        "within_iqr_max_price",
        "within_iqr_spread"
    )
    .orderBy("item_slug", "order_type")
)

In [0]:
# Validate summary grain and aggregated totals
summary_validation = market_summary.agg(
    F.count("*").alias("total_groups"),
    F.countDistinct(
        "execution_id", "item_slug", "order_type"
    ).alias("unique_groups"),
    F.sum("total_offers").alias("total_offers"),
    F.sum("within_iqr_offers").alias("within_iqr_offers"),
    F.sum("outlier_offers").alias("outlier_offers"),
    F.sum("insufficient_sample_offers").alias("insufficient_sample_offers")
)

display(summary_validation)

In [0]:
# Define Gold target table
MART_TARGET = get_gold_table(
    "marts",
    "mart_item_price_summary"
)

# Materialize market summary
(
    market_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(MART_TARGET)
)

print(f"Table successfully materialized: {MART_TARGET}")

In [0]:
# Read materialized Gold table
saved_summary = spark.table(MART_TARGET)

# Validate persisted records
display(
    saved_summary.agg(
        F.count("*").alias("total_groups"),
        F.countDistinct(
            "execution_id", "item_slug", "order_type"
        ).alias("unique_groups"),
        F.sum("total_offers").alias("total_offers"),
        F.sum("within_iqr_offers").alias("within_iqr_offers"),
        F.sum("outlier_offers").alias("outlier_offers"),
        F.sum("insufficient_sample_offers").alias("insufficient_sample_offers")
    )
)